# HRNN on MovieLens 1M

End-to-end demo of `HRNNEstimator` on the MovieLens 1M dataset.

**Model**: Hierarchical Recurrent Neural Network (Quadrana et al., RecSys 2017).  
Two GRU levels: a **session GRU** captures short-term within-session dynamics; a **user GRU** carries long-term preference context across sessions. The user GRU state seeds the session GRU's `h_0` at the start of each new session.

**Session detection**: no `SESSION_ID` column in ML-1M, so sessions are inferred from timestamp gaps (≥ 24 hours of inactivity → new session).

**Evaluation protocol**: leave-last-out on positive interactions (rating ≥ 4). For each user the last positive interaction is the test item; everything prior is training history.  
**Metrics**: HR@10 and NDCG@10 on sampled ranking (1 positive + 100 random negatives).

**Sanity checks included**:
1. Session structure — verify sessions are built correctly before training
2. Training success — loss is non-trivial and model is fitted
3. Score distribution — scores vary meaningfully across items
4. Personalization — different users receive different recommendations
5. Item embeddings — genre-similar movies have higher cosine similarity than genre-dissimilar pairs

## 1. Imports

In [1]:
import logging
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.estimator.sequential import HRNNClassifierEstimator
from skrec.recommender.sequential import HierarchicalSequentialRecommender
from skrec.scorer.hierarchical import HierarchicalScorer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

RAW_DIR = Path("data/movielens-1m")
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/hrnn")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

Imports OK


## 2. Download MovieLens 1M

In [2]:
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
zip_path = RAW_DIR / "ml-1m.zip"

if not (RAW_DIR / "ratings.dat").exists():
    print("Downloading MovieLens 1M...")
    urllib.request.urlretrieve(ML1M_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith(".dat"):
                filename = Path(name).name
                with zf.open(name) as src, open(RAW_DIR / filename, "wb") as dst:
                    dst.write(src.read())
    print("Downloaded and extracted.")
else:
    print("Already downloaded.")

Already downloaded.


## 3. Load and Preprocess

In [3]:
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
)

movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1",
)

print(f"Ratings: {len(ratings):,}  |  Users: {ratings.UserID.nunique():,}  |  Movies: {ratings.MovieID.nunique():,}")
ratings.head()

Ratings: 1,000,209  |  Users: 6,040  |  Movies: 3,706


,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


### Choosing your OUTCOME column: four training modes

HRNN (and SASRec) are flexible about what the OUTCOME column represents. The choice determines how the model interprets the reward signal and which estimator class to use.

| Mode | OUTCOME value | Estimator | `num_negatives` | When to use |
|---|---|---|---|---|
| **Binary (positives only)** | `rating >= 4 → 1.0`, drop rest | `HRNNClassifierEstimator` | `>= 1` required | Clean positive signal; random sampling is the only source of negatives. |
| **Binary with explicit negatives** | `rating >= 4 → 1.0`, `rating <= 2 → 0.0`, drop 3 | `HRNNClassifierEstimator` | `>= 1` still useful | Dislikes provide hard negatives; random negatives still needed so the model learns baseline scores for unseen items. |
| **Soft-label** | `rating / 5.0` → `[0.2, 1.0]` | `HRNNClassifierEstimator` (BCE) | `>= 1` | Rating as target probability. |
| **Regression** | `rating` as-is (`1–5`) | `HRNNRegressorEstimator` (MSE) | `>= 1` | Model predicts rating directly. |

**This notebook uses mode 2 (binary with explicit negatives).** Two things change vs mode 1:
1. **Data prep** — keep `rating <= 2` with `OUTCOME=0.0` instead of dropping them.
2. **Estimator** — `num_negatives >= 1` is still needed. Explicit dislikes teach the model about specific rejected items; random negatives teach it the baseline score for the vast majority of items the user has never seen. Setting `num_negatives=0` collapses performance below random because unseen items receive arbitrary scores with no gradient.

> The scorer and recommender are identical across all modes.

In [4]:
# Build interactions with explicit positive and negative outcomes.
#
# rating >= 4  →  OUTCOME = 1.0  (liked)
# rating <= 2  →  OUTCOME = 0.0  (disliked — explicit hard negative)
# rating == 3  →  dropped        (neutral/ambiguous)
#
# Keeping dislikes in the sequence gives the model a real signal about
# what the user watched and rejected, which is stronger than randomly
# sampled items the user has never seen.
# With explicit negatives present, we set num_negatives=0 at training time.

pos_mask = ratings["Rating"] >= 4
neg_mask = ratings["Rating"] <= 2

pos_df = ratings[pos_mask].copy()
neg_df = ratings[neg_mask].copy()

interactions = pd.concat(
    [
        pd.DataFrame(
            {
                "USER_ID": pos_df["UserID"].astype(str),
                "ITEM_ID": pos_df["MovieID"].astype(str),
                "OUTCOME": 1.0,
                "TIMESTAMP": pos_df["Timestamp"],
            }
        ),
        pd.DataFrame(
            {
                "USER_ID": neg_df["UserID"].astype(str),
                "ITEM_ID": neg_df["MovieID"].astype(str),
                "OUTCOME": 0.0,
                "TIMESTAMP": neg_df["Timestamp"],
            }
        ),
    ],
    ignore_index=True,
)

items = pd.DataFrame({"ITEM_ID": movies["MovieID"].astype(str)})

print(f"Positive interactions (rating >= 4) : {pos_mask.sum():,}")
print(f"Negative interactions (rating <= 2) : {neg_mask.sum():,}")
print(f"Neutral dropped  (rating == 3)      : {(ratings['Rating'] == 3).sum():,}")
print(f"Total kept                           : {len(interactions):,}")
interactions.head()

Positive interactions (rating >= 4) : 575,281
Negative interactions (rating <= 2) : 163,731
Neutral dropped  (rating == 3)      : 261,197
Total kept                           : 739,012


,USER_ID,ITEM_ID,OUTCOME,TIMESTAMP
0,1,1193,1.0,978300760
1,1,3408,1.0,978300275
2,1,2355,1.0,978824291
3,1,1287,1.0,978302039
4,1,2804,1.0,978300719


## 4. Train / Test Split (Leave-Last-Out on Positive Interactions)

For each user the **last** positive interaction is the test item; everything before it
is training history (including the second-to-last, which serves as the last training target).
Users with fewer than 5 positive interactions are excluded.

**Evaluation**: sampled ranking — test item ranked against 100 randomly drawn negatives.

In [5]:
interactions = interactions.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

user_counts = interactions.groupby("USER_ID").size()
valid_users = user_counts[user_counts >= 5].index
interactions = interactions[interactions["USER_ID"].isin(valid_users)].reset_index(drop=True)

# Leave-last-out on POSITIVE interactions only.
# The test item must be a liked item — we evaluate whether the model can
# rank a positive recommendation highly, not whether it can predict a dislike.
positives = interactions[interactions["OUTCOME"] == 1.0].copy()
positives["rank"] = positives.groupby("USER_ID").cumcount(ascending=False)
test_df = positives[positives["rank"] == 0].drop(columns=["rank"]).reset_index(drop=True)

# Train on ALL interactions (positives + negatives, including the test item).
train_df = interactions.copy()

# Eval history: all interactions except the specific test interaction.
all_except_test_df = (
    interactions.merge(
        test_df[["USER_ID", "ITEM_ID", "TIMESTAMP"]],
        on=["USER_ID", "ITEM_ID", "TIMESTAMP"],
        how="left",
        indicator=True,
    )
    .query('_merge == "left_only"')
    .drop(columns=["_merge"])
    .reset_index(drop=True)
)

print(f"Train interactions : {len(train_df):,}")
print(f"Test  interactions : {len(test_df):,}  (last positive per user)")
print(f"Users              : {train_df.USER_ID.nunique():,}")

Train interactions : 739,012
Test  interactions : 6,038  (last positive per user)
Users              : 6,040


## 5. Sanity Check: Session Structure

Before training, verify that the session detection logic produces a sensible structure.
ML-1M has no explicit `SESSION_ID`, so sessions are inferred from timestamp gaps.
We use a **24-hour (1440 min) timeout**: a gap ≥ 24 hours between consecutive ratings
by the same user starts a new session. This treats each distinct viewing day as one session.

In [6]:
SESSION_TIMEOUT_MINUTES = 1440  # 24 hours
MAX_SESSIONS = 15
MAX_SESSION_LEN = 30


# Build session sequences from train_df using the same logic the recommender will use.
# We do this manually here just for inspection — the recommender handles it internally.
def _detect_sessions(df, timeout_minutes):
    df = df.sort_values(["USER_ID", "TIMESTAMP"]).copy()
    timeout_sec = timeout_minutes * 60
    gap = df.groupby("USER_ID")["TIMESTAMP"].diff().fillna(float("inf"))
    df["new_session"] = gap > timeout_sec
    df["session_num"] = df.groupby("USER_ID")["new_session"].cumsum()
    return df


train_with_sessions = _detect_sessions(train_df, SESSION_TIMEOUT_MINUTES)

# Per-user stats
session_stats = train_with_sessions.groupby(["USER_ID", "session_num"]).size().rename("session_len").reset_index()
sessions_per_user = session_stats.groupby("USER_ID")["session_num"].count()
items_per_session = session_stats["session_len"]

print("=== Session Structure (train set) ===")
print(f"Total sessions detected : {len(session_stats):,}")
print("")
print("Sessions per user:")
print(f"  median : {sessions_per_user.median():.0f}")
print(f"  mean   : {sessions_per_user.mean():.1f}")
print(f"  max    : {sessions_per_user.max()}")
print(f"  90th%  : {sessions_per_user.quantile(0.9):.0f}")
print("")
print("Items per session:")
print(f"  median : {items_per_session.median():.0f}")
print(f"  mean   : {items_per_session.mean():.1f}")
print(f"  max    : {items_per_session.max()}")
print(f"  90th%  : {items_per_session.quantile(0.9):.0f}")
print("")
print(
    f"Single-item sessions    : {(items_per_session == 1).sum():,}  "
    f"({(items_per_session == 1).mean():.1%} of all sessions)"
)
print("")
print(f"Model config  →  max_sessions={MAX_SESSIONS}, max_session_len={MAX_SESSION_LEN}")
coverage = (sessions_per_user <= MAX_SESSIONS).mean()
print(f"Users whose full history fits in max_sessions : {coverage:.1%}")

=== Session Structure (train set) ===
Total sessions detected : 19,162

Sessions per user:
  median : 1
  mean   : 3.2
  max    : 182
  90th%  : 6

Items per session:
  median : 7
  mean   : 38.6
  max    : 1442
  90th%  : 112

Single-item sessions    : 5,007  (26.1% of all sessions)

Model config  →  max_sessions=15, max_session_len=30
Users whose full history fits in max_sessions : 96.1%


In [7]:
# Show one user's session structure as a concrete example
movie_title = movies.set_index(movies["MovieID"].astype(str))["Title"].to_dict()

sample_user = train_with_sessions["USER_ID"].iloc[0]
user_sessions = train_with_sessions[train_with_sessions["USER_ID"] == sample_user]

print(f"User {sample_user} — session breakdown (first 5 sessions):")
for sess_id, grp in list(user_sessions.groupby("session_num"))[:5]:
    titles = [movie_title.get(iid, iid) for iid in grp["ITEM_ID"]]
    ts_range = f"{grp['TIMESTAMP'].min()} – {grp['TIMESTAMP'].max()}"
    print(f"  Session {int(sess_id):2d} ({len(grp)} item{'s' if len(grp) > 1 else ''}, ts={ts_range}):")
    for t in titles:
        print(f"    · {t}")

User 1 — session breakdown (first 5 sessions):
  Session  1 (34 items, ts=978300019 – 978302281):
    · Girl, Interrupted (1999)
    · Back to the Future (1985)
    · Titanic (1997)
    · Cinderella (1950)
    · Last Days of Disco, The (1998)
    · Erin Brockovich (2000)
    · Christmas Story, A (1983)
    · To Kill a Mockingbird (1962)
    · One Flew Over the Cuckoo's Nest (1975)
    · Star Wars: Episode IV - A New Hope (1977)
    · Wizard of Oz, The (1939)
    · Fargo (1996)
    · Run Lola Run (Lola rennt) (1998)
    · Rain Man (1988)
    · Saving Private Ryan (1998)
    · Awakenings (1990)
    · Gigi (1958)
    · Sound of Music, The (1965)
    · Driving Miss Daisy (1989)
    · Bambi (1942)
    · Apollo 13 (1995)
    · Mary Poppins (1964)
    · E.T. the Extra-Terrestrial (1982)
    · Ben-Hur (1959)
    · Big (1988)
    · Sixth Sense, The (1999)
    · Dead Poets Society (1989)
    · Ferris Bueller's Day Off (1986)
    · Secret Garden, The (1993)
    · Toy Story 2 (1999)
    · Airplane

## 6. Save CSVs and Create Datasets

In [8]:
train_path = str(DATA_DIR / "train_interactions.csv")
valid_path = str(DATA_DIR / "valid_interactions.csv")
items_path = str(DATA_DIR / "items.csv")

# Second-to-last positive per user — used as early-stopping validation signal.
# `positives` still carries the `rank` column from the split cell.
valid_df = positives[positives["rank"] == 1].drop(columns=["rank"]).reset_index(drop=True)

if not Path(train_path).exists():
    train_df.to_csv(train_path, index=False)
if not Path(valid_path).exists():
    valid_df.to_csv(valid_path, index=False)
if not Path(items_path).exists():
    items.to_csv(items_path, index=False)

interactions_ds = InteractionsDataset(data_location=train_path)
valid_inter_ds = InteractionsDataset(data_location=valid_path)
items_ds = ItemsDataset(data_location=items_path)

print(f"Training data   : {len(train_df):,} interactions saved to {train_path}")
print(f"Validation data : {len(valid_df):,} interactions (second-to-last positive per user)")
print("Datasets created.")

Training data   : 739,012 interactions saved to data/hrnn/train_interactions.csv


Validation data : 6,037 interactions (second-to-last positive per user)
Datasets created.


## 7. Build and Train HRNN

Key hyperparameters:
- `hidden_units=50` — embedding and GRU hidden dimension (same as SASRec baseline)
- `num_layers=1` — single-layer GRUs as in the original HRNN paper
- `max_sessions=15` — retain last 15 sessions per user
- `max_session_len=30` — up to 30 items per session
- `session_timeout_minutes=1440` — 24-hour gap → new session
- `early_stopping_patience=5` — stop if validation session-loss doesn't improve for 5 epochs;
  `restore_best_weights=True` reverts to the best checkpoint automatically

In [9]:
estimator = HRNNClassifierEstimator(
    hidden_units=50,
    num_layers=1,
    dropout_rate=0.2,
    num_negatives=1,  # explicit dislikes + 1 random negative for unseen-item coverage
    max_sessions=MAX_SESSIONS,
    max_session_len=MAX_SESSION_LEN,
    learning_rate=0.001,
    epochs=100,
    batch_size=128,
    optimizer_name="adam",
    loss_fn_name="bce",
    early_stopping_patience=5,  # stop if val loss doesn't improve for 5 epochs
    restore_best_weights=True,
    random_state=42,
    verbose=1,
)

scorer = HierarchicalScorer(estimator)
recommender = HierarchicalSequentialRecommender(
    scorer,
    max_sessions=MAX_SESSIONS,
    max_session_len=MAX_SESSION_LEN,
    session_timeout_minutes=SESSION_TIMEOUT_MINUTES,
)

print("Training HRNN...")
recommender.train(
    items_ds=items_ds,
    interactions_ds=interactions_ds,
    use_validation=True,
)
print("Training complete.")

Training HRNN...


2026-04-17 10:20:47,848 - skrec.recommender.sequential.hierarchical_recommender - INFO Detected session boundaries using timeout=1440 minutes.


2026-04-17 10:20:47,848 skrec.recommender.sequential.hierarchical_recommender INFO Detected session boundaries using timeout=1440 minutes.


2026-04-17 10:20:48,187 - skrec.recommender.sequential.hierarchical_recommender - INFO Built session sequences for 6040 users (max_sessions=15, max_session_len=30, has_outcome=True).


2026-04-17 10:20:48,187 skrec.recommender.sequential.hierarchical_recommender INFO Built session sequences for 6040 users (max_sessions=15, max_session_len=30, has_outcome=True).


...

2026-04-17 10:33:45,979 - skrec.estimator.sequential.hrnn_estimator - INFO Epoch [99/100], Loss: 0.7285, Val Loss: 0.7290


2026-04-17 10:33:45,979 skrec.estimator.sequential.hrnn_estimator INFO Epoch [99/100], Loss: 0.7285, Val Loss: 0.7290


2026-04-17 10:33:53,826 - skrec.estimator.sequential.hrnn_estimator - INFO Epoch [100/100], Loss: 0.7286, Val Loss: 0.7269


2026-04-17 10:33:53,826 skrec.estimator.sequential.hrnn_estimator INFO Epoch [100/100], Loss: 0.7286, Val Loss: 0.7269


Training complete.


## 8. Sanity Check: Training Success

Verify that the model was fitted and the internal state is consistent.

In [10]:
import torch

assert estimator.model is not None, "Model was not built — training failed silently."

n_params = sum(p.numel() for p in estimator.model.parameters())
item_emb_weight = estimator.model.item_embedding.weight.data

# Item embedding should NOT be all-zero after training
emb_norm = item_emb_weight[1:].norm(dim=1)  # skip padding at index 0
assert emb_norm.mean().item() > 0.01, "Item embeddings are near-zero — training may not have updated weights."

# Padding embedding (index 0) must stay zero
assert item_emb_weight[0].abs().max().item() == 0.0, "Padding embedding (index 0) is non-zero."

# Item vocab size should match items_df
assert estimator.num_items == len(estimator.item_id_index), "Item vocab size mismatch."

print("=== Training Sanity Check ===")
print(f"Model parameters        : {n_params:,}")
print(f"Item vocabulary size    : {estimator.num_items:,}")
print(f"Users in training       : {estimator.num_users:,}")
print(f"Embedding norm (mean)   : {emb_norm.mean().item():.4f}")
print(f"Embedding norm (std)    : {emb_norm.std().item():.4f}")
print(f"Padding embedding norm  : {item_emb_weight[0].norm().item():.6f}  (must be 0)")
print(f"Device                  : {estimator.device}")
print()
print("All training sanity checks passed.")

=== Training Sanity Check ===
Model parameters        : 224,800
Item vocabulary size    : 3,883
Users in training       : 6,040
Embedding norm (mean)   : 1.2846
Embedding norm (std)    : 0.4412
Padding embedding norm  : 0.000000  (must be 0)
Device                  : cpu

All training sanity checks passed.


## 9. Evaluate: HR@10 and NDCG@10 (Sampled Ranking)

For each user the held-out test item is ranked against **100 randomly sampled negative
items** (movies the user has not rated). HR@10 and NDCG@10 are computed within these
101 candidates.

In [11]:
rng = np.random.default_rng(42)
all_item_ids = np.array(list(scorer.item_names))

known_items = set(scorer.item_names)
eval_test_df = test_df[test_df["ITEM_ID"].isin(known_items)].copy()
eval_users = set(eval_test_df["USER_ID"])

eval_history_df = all_except_test_df[all_except_test_df["USER_ID"].isin(eval_users)].copy()
eval_history_df = eval_history_df.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

print(f"Evaluating {len(eval_users):,} users (sampled ranking: 1 positive + 100 negatives)...")

# Build session sequences for evaluation users (same session timeout as training)
sessions_df = recommender._build_session_sequences(eval_history_df)
user_order = sessions_df["USER_ID"].tolist()

# Get (num_users, num_items) score matrix in one forward pass
all_scores = estimator.predict_proba_with_embeddings(interactions=sessions_df)

item_name_to_idx = {name: i for i, name in enumerate(scorer.item_names)}
gt_lookup = eval_test_df.set_index("USER_ID")["ITEM_ID"].to_dict()
user_items = interactions.groupby("USER_ID")["ITEM_ID"].apply(set).to_dict()

TOP_K = 10
N_NEGATIVES = 100

hits, ndcgs = [], []
for i, user_id in enumerate(user_order):
    test_item = gt_lookup.get(user_id)
    if test_item is None or test_item not in item_name_to_idx:
        continue

    seen = user_items.get(user_id, set())
    candidates = all_item_ids[~np.isin(all_item_ids, list(seen))]
    neg_sample = rng.choice(candidates, size=min(N_NEGATIVES, len(candidates)), replace=False)

    candidate_ids = [test_item] + list(neg_sample)
    candidate_idxs = [item_name_to_idx[c] for c in candidate_ids if c in item_name_to_idx]
    test_score = all_scores[i, item_name_to_idx[test_item]]
    rank = int((all_scores[i, candidate_idxs] > test_score).sum()) + 1

    hits.append(1 if rank <= TOP_K else 0)
    ndcgs.append(1.0 / np.log2(rank + 1) if rank <= TOP_K else 0.0)

print(f"\n{'=' * 40}")
print(f"Evaluation: 1 positive + {N_NEGATIVES} random negatives")
print(f"HR@{TOP_K}   : {np.mean(hits):.4f}")
print(f"NDCG@{TOP_K} : {np.mean(ndcgs):.4f}")
print(f"Users evaluated: {len(hits):,}")
print(f"{'=' * 40}")

Evaluating 6,038 users (sampled ranking: 1 positive + 100 negatives)...


2026-04-17 10:33:54,084 - skrec.recommender.sequential.hierarchical_recommender - INFO Detected session boundaries using timeout=1440 minutes.


2026-04-17 10:33:54,084 skrec.recommender.sequential.hierarchical_recommender INFO Detected session boundaries using timeout=1440 minutes.


2026-04-17 10:33:54,453 - skrec.recommender.sequential.hierarchical_recommender - INFO Built session sequences for 6038 users (max_sessions=15, max_session_len=30, has_outcome=True).


2026-04-17 10:33:54,453 skrec.recommender.sequential.hierarchical_recommender INFO Built session sequences for 6038 users (max_sessions=15, max_session_len=30, has_outcome=True).



Evaluation: 1 positive + 100 random negatives
HR@10   : 0.6794
NDCG@10 : 0.3958
Users evaluated: 6,038


## 10. Sanity Check: Score Distribution

If training collapsed (all items receive identical scores), every ranking is random and
HR@10 ≈ 10/101 ≈ 0.099. Verify that score variance is non-trivial and that the
inter-user variance (do different users get different score vectors?) is also non-zero.

In [12]:
# all_scores: (num_eval_users, num_items) computed in the evaluation cell above

# 1. Within-user score variance — should be >> 0
within_user_std = all_scores.std(axis=1)  # std across items for each user

# 2. Cross-user variance — different users should have different score vectors
cross_user_std = all_scores.std(axis=0)  # std across users for each item

# 3. Score range
global_min = all_scores.min()
global_max = all_scores.max()

# 4. Degenerate check: how many users have effectively constant scores (std < 0.01)?
n_degenerate = (within_user_std < 0.01).sum()

print("=== Score Distribution Sanity Check ===")
print(f"Score matrix shape          : {all_scores.shape}")
print(f"Global score range          : [{global_min:.4f}, {global_max:.4f}]")
print("")
print("Within-user score std (std across items per user):")
print(f"  mean : {within_user_std.mean():.4f}")
print(f"  min  : {within_user_std.min():.4f}")
print(f"  max  : {within_user_std.max():.4f}")
print("")
print("Cross-user item score std (std across users per item):")
print(f"  mean : {cross_user_std.mean():.4f}")
print(f"  min  : {cross_user_std.min():.4f}")
print("")
print(f"Degenerate users (within-std < 0.01) : {n_degenerate} / {len(within_user_std)}")

assert within_user_std.mean() > 0.01, "Score variance is near-zero — model outputs are collapsed."
assert n_degenerate < len(within_user_std) * 0.05, "More than 5% of users have degenerate scores."

random_hr_baseline = TOP_K / (N_NEGATIVES + 1)
assert np.mean(hits) > random_hr_baseline * 1.5, (
    f"HR@{TOP_K}={np.mean(hits):.4f} is not meaningfully above random baseline {random_hr_baseline:.3f}."
)
print()
print(f"HR@10 ({np.mean(hits):.4f}) > 1.5x random baseline ({random_hr_baseline:.3f}): OK")
print("All score distribution checks passed.")

=== Score Distribution Sanity Check ===
Score matrix shape          : (6038, 3883)
Global score range          : [-17.0798, 3.9934]

Within-user score std (std across items per user):
  mean : 3.1986
  min  : 1.5710
  max  : 4.2115

Cross-user item score std (std across users per item):
  mean : 1.3583
  min  : 0.3231

Degenerate users (within-std < 0.01) : 0 / 6038

HR@10 (0.6794) > 1.5x random baseline (0.099): OK
All score distribution checks passed.


## 11. Sanity Check: Personalization

A well-trained model should produce *different* score vectors for users with different
tastes. We:
1. Find a user whose history is predominantly **Animation** and one predominantly **Action**.
2. Compute the cosine similarity between their score vectors — should be well below 1.
3. Confirm their top-10 recommendations are distinct.

In [13]:
# Build a genre lookup: MovieID -> set of genres
movie_genres = movies.set_index(movies["MovieID"].astype(str))["Genres"].str.split("|").to_dict()


def dominant_genre(user_id, history_df, genre_lookup):
    """Return the most-watched genre for a user."""
    items_watched = history_df[history_df["USER_ID"] == user_id]["ITEM_ID"].tolist()
    from collections import Counter

    counts = Counter(g for iid in items_watched for g in genre_lookup.get(iid, []))
    return counts.most_common(1)[0] if counts else ("Unknown", 0)


# Find users with at least 10 interactions in eval set
eval_user_list = list(eval_users)
user_interaction_counts = eval_history_df.groupby("USER_ID").size()
active_users = user_interaction_counts[user_interaction_counts >= 10].index

# Score dominant genres for a sample of active users
genre_profiles = {}
for uid in list(active_users)[:300]:  # sample 300 users for speed
    genre, count = dominant_genre(uid, eval_history_df, movie_genres)
    genre_profiles[uid] = (genre, count)

# Pick one Animation-dominant and one Action-dominant user
animation_users = [(uid, cnt) for uid, (g, cnt) in genre_profiles.items() if g == "Animation"]
action_users = [(uid, cnt) for uid, (g, cnt) in genre_profiles.items() if g == "Action"]

if not animation_users or not action_users:
    # Fallback: pick any two users with low score-vector similarity
    print("Not enough genre-specific users found; picking arbitrary users for comparison.")
    user_a_id, user_b_id = eval_user_list[0], eval_user_list[1]
else:
    user_a_id = max(animation_users, key=lambda x: x[1])[0]  # most animation-heavy
    user_b_id = max(action_users, key=lambda x: x[1])[0]  # most action-heavy

# Get score vectors for these two users
idx_a = user_order.index(user_a_id) if user_a_id in user_order else None
idx_b = user_order.index(user_b_id) if user_b_id in user_order else None

if idx_a is None or idx_b is None:
    print("One of the selected users not in eval set — picking first two eval users.")
    idx_a, idx_b = 0, 1
    user_a_id, user_b_id = user_order[0], user_order[1]

vec_a = all_scores[idx_a]
vec_b = all_scores[idx_b]

# Cosine similarity between the two score vectors
cos_sim = float(np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b) + 1e-9))

# Top-10 recommendations for each user
top10_a = list(np.array(list(scorer.item_names))[np.argsort(vec_a)[::-1][:10]])
top10_b = list(np.array(list(scorer.item_names))[np.argsort(vec_b)[::-1][:10]])
overlap = len(set(top10_a) & set(top10_b))

genre_a, cnt_a = genre_profiles.get(user_a_id, ("?", 0))
genre_b, cnt_b = genre_profiles.get(user_b_id, ("?", 0))

print("=== Personalization Sanity Check ===")
print(f"User A ({user_a_id}) — dominant genre: {genre_a} ({cnt_a} interactions)")
print(f"User B ({user_b_id}) — dominant genre: {genre_b} ({cnt_b} interactions)")
print("")
print(f"Score vector cosine similarity : {cos_sim:.4f}  (lower = more personalized)")
print(f"Top-10 recommendation overlap  : {overlap}/10")
print()
print("User A top-10:")
for iid in top10_a:
    print(f"  · {movie_title.get(iid, iid)}")
print("\nUser B top-10:")
for iid in top10_b:
    print(f"  · {movie_title.get(iid, iid)}")

assert cos_sim < 0.999, "Score vectors are identical — model is not personalized."
assert overlap < 10, "Top-10 lists are completely identical — model is not personalized."
print()
print("Personalization check passed.")

=== Personalization Sanity Check ===
User A (112) — dominant genre: Animation (27 interactions)
User B (1244) — dominant genre: Action (181 interactions)

Score vector cosine similarity : 0.9715  (lower = more personalized)
Top-10 recommendation overlap  : 1/10

User A top-10:
  · Rear Window (1954)
  · Bridge on the River Kwai, The (1957)
  · Vertigo (1958)
  · Casablanca (1942)
  · Graduate, The (1967)
  · Hustler, The (1961)
  · Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)
  · Bonnie and Clyde (1967)
  · To Kill a Mockingbird (1962)
  · Singin' in the Rain (1952)

User B top-10:
  · Wonder Boys (2000)
  · Almost Famous (2000)
  · Rear Window (1954)
  · North by Northwest (1959)
  · Requiem for a Dream (2000)
  · Hoop Dreams (1994)
  · Godfather, The (1972)
  · Croupier (1998)
  · Annie Hall (1977)
  · Best in Show (2000)

Personalization check passed.


## 12. Sanity Check: Item Embeddings

After training, the item embedding matrix should reflect genre/content similarity.
We check that the **average cosine similarity between same-genre movie pairs** is
higher than between **cross-genre pairs**.

We use Animation vs. Action as two well-separated genres.

In [14]:
import torch.nn.functional as F

# Extract item embeddings (skip index 0 = padding)
with torch.no_grad():
    emb_matrix = estimator.model.item_embedding.weight[1:].cpu()  # (num_items, H)
    emb_norm = F.normalize(emb_matrix, dim=1)  # unit vectors

# Map item_id_index position → embedding row (item_id_index[i] → row i in emb_matrix)
item_to_emb_row = {iid: i for i, iid in enumerate(estimator.item_id_index)}


def genre_item_ids(genre, movie_genres, item_to_emb_row, min_items=20):
    """Return embedding row indices for items of a given genre that are in the trained vocab."""
    rows = [item_to_emb_row[iid] for iid, genres in movie_genres.items() if genre in genres and iid in item_to_emb_row]
    return rows


anim_rows = genre_item_ids("Animation", movie_genres, item_to_emb_row)
action_rows = genre_item_ids("Action", movie_genres, item_to_emb_row)

# Sample up to 50 items per genre for tractable pairwise computation
rng2 = np.random.default_rng(0)
anim_sample = rng2.choice(anim_rows, size=min(50, len(anim_rows)), replace=False)
action_sample = rng2.choice(action_rows, size=min(50, len(action_rows)), replace=False)

anim_emb = emb_norm[anim_sample]  # (Na, H)
action_emb = emb_norm[action_sample]  # (Nb, H)

# Within-Animation similarity
sim_anim_anim = (anim_emb @ anim_emb.T).numpy()
np.fill_diagonal(sim_anim_anim, np.nan)  # exclude self-similarity
mean_anim_anim = np.nanmean(sim_anim_anim)

# Within-Action similarity
sim_act_act = (action_emb @ action_emb.T).numpy()
np.fill_diagonal(sim_act_act, np.nan)
mean_act_act = np.nanmean(sim_act_act)

# Cross-genre similarity
sim_anim_act = (anim_emb @ action_emb.T).numpy()
mean_cross = np.nanmean(sim_anim_act)

print("=== Item Embedding Sanity Check ===")
print(f"Animation movies in vocab : {len(anim_rows)}  (sampled {len(anim_sample)})")
print(f"Action movies in vocab    : {len(action_rows)}  (sampled {len(action_sample)})")
print()
print(f"Within-Animation cosine similarity (mean) : {mean_anim_anim:.4f}")
print(f"Within-Action    cosine similarity (mean) : {mean_act_act:.4f}")
print(f"Cross-genre      cosine similarity (mean) : {mean_cross:.4f}")
print()

within_avg = (mean_anim_anim + mean_act_act) / 2
if within_avg > mean_cross:
    print(f"Within-genre similarity ({within_avg:.4f}) > cross-genre ({mean_cross:.4f}) ✓")
    print("Item embeddings cluster by genre — training learned meaningful representations.")
else:
    print(f"Within-genre similarity ({within_avg:.4f}) <= cross-genre ({mean_cross:.4f})")
    print("Note: genre clustering is a soft check — HRNN is a sequential model that")
    print("learns co-occurrence patterns, which may not align perfectly with genre labels.")
    print("The HR@10 metric above is a stronger indicator of model quality.")

# Spot-check: find the nearest neighbor of a known Animation movie
# Toy Story = MovieID 1 in ML-1M
toy_story_id = "1"
if toy_story_id in item_to_emb_row:
    ts_emb = emb_norm[item_to_emb_row[toy_story_id]].unsqueeze(0)  # (1, H)
    sims = (ts_emb @ emb_norm.T).squeeze().numpy()  # (num_items,)
    sims[item_to_emb_row[toy_story_id]] = -1  # exclude self
    top5_nn = np.argsort(sims)[::-1][:5]
    top5_ids = [estimator.item_id_index[j] for j in top5_nn]
    print("\nNearest neighbours of 'Toy Story (1995)' by embedding similarity:")
    for iid in top5_ids:
        genres = "|".join(movie_genres.get(iid, []))
        print(f"  · {movie_title.get(iid, iid)}  [{genres}]")

=== Item Embedding Sanity Check ===
Animation movies in vocab : 105  (sampled 50)
Action movies in vocab    : 503  (sampled 50)

Within-Animation cosine similarity (mean) : 0.2317
Within-Action    cosine similarity (mean) : 0.2797
Cross-genre      cosine similarity (mean) : 0.1986

Within-genre similarity (0.2557) > cross-genre (0.1986) ✓
Item embeddings cluster by genre — training learned meaningful representations.

Nearest neighbours of 'Toy Story (1995)' by embedding similarity:
  · Back to the Future (1985)  [Comedy|Sci-Fi]
  · Airplane! (1980)  [Comedy]
  · Usual Suspects, The (1995)  [Crime|Thriller]
  · When Harry Met Sally... (1989)  [Comedy|Romance]
  · Apocalypse Now (1979)  [Drama|War]


## 13. Sample Recommendations

Qualitative inspection of top-10 recommendations for a few users alongside their held-out test item.
Recommendations are from the full-item ranking (all items scored, not sampled).

In [15]:
top_k_recs = recommender.recommend(interactions=eval_history_df, top_k=TOP_K)

sample_users = user_order[:5]
for user_id in sample_users:
    idx = user_order.index(user_id)
    recs = list(top_k_recs[idx])
    test_item = gt_lookup.get(user_id, "?")
    hit = "HIT" if test_item in recs else "MISS"
    dom_genre, _ = genre_profiles.get(user_id, ("?", 0))
    n_sessions_user = train_with_sessions[train_with_sessions["USER_ID"] == user_id]["session_num"].nunique()

    print(f"\nUser {user_id}  |  Genre profile: {dom_genre}  |  Sessions: {n_sessions_user}")
    print(f"  Test item : {movie_title.get(test_item, test_item)}  [{hit}]")
    print("  Top-10 (full-item ranking):")
    for rank, item_id in enumerate(recs, 1):
        marker = " <-- TEST ITEM" if item_id == test_item else ""
        print(f"    {rank:2}. {movie_title.get(item_id, item_id)}{marker}")

2026-04-17 10:34:03,528 - skrec.recommender.sequential.hierarchical_recommender - INFO Detected session boundaries using timeout=1440 minutes.


2026-04-17 10:34:03,528 skrec.recommender.sequential.hierarchical_recommender INFO Detected session boundaries using timeout=1440 minutes.


2026-04-17 10:34:03,930 - skrec.recommender.sequential.hierarchical_recommender - INFO Built session sequences for 6038 users (max_sessions=15, max_session_len=30, has_outcome=True).


2026-04-17 10:34:03,930 skrec.recommender.sequential.hierarchical_recommender INFO Built session sequences for 6038 users (max_sessions=15, max_session_len=30, has_outcome=True).



User 1  |  Genre profile: Drama  |  Sessions: 2
  Test item : Pocahontas (1995)  [MISS]
  Top-10 (full-item ranking):
     1. Sleepless in Seattle (1993)
     2. Dave (1993)
     3. Mulan (1998)
     4. Pretty Woman (1990)
     5. Interview with the Vampire (1994)
     6. Patch Adams (1998)
     7. Mrs. Doubtfire (1993)
     8. Forrest Gump (1994)
     9. Liar Liar (1997)
    10. Top Gun (1986)

User 10  |  Genre profile: Comedy  |  Sessions: 4
  Test item : Hero (1992)  [MISS]
  Top-10 (full-item ranking):
     1. Shawshank Redemption, The (1994)
     2. Annie Hall (1977)
     3. Apollo 13 (1995)
     4. Joy Luck Club, The (1993)
     5. Rain Man (1988)
     6. Rear Window (1954)
     7. Wizard of Oz, The (1939)
     8. Paper Chase, The (1973)
     9. When Harry Met Sally... (1989)
    10. Casablanca (1942)

User 100  |  Genre profile: Action  |  Sessions: 1
  Test item : Wizard of Oz, The (1939)  [MISS]
  Top-10 (full-item ranking):
     1. Silence of the Lambs, The (1991)
     2. S